# Notebook 02 — Input Data Structures and Loading

This notebook walks through every input file and the functions that parse them.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_DIR = PROJECT_ROOT / "notebooks" / "sample_data"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1 · protephospho.csv — dual-row format

In [ ]:
df_pp = pd.read_csv(SAMPLE_DIR / "protephospho.csv")
print("Shape:", df_pp.shape)
df_pp.head(8)


### Dual-row format explained

Each protein appears **twice** per row type:

* **Protein row** — `Psite` is empty (`NaN`); values are the total protein abundance
* **Phosphosite row** — `Psite` is e.g. `"Y1068"`; values are site-specific intensities

`load_site_data()` separates these automatically.


In [ ]:
# Show protein rows vs phosphosite rows side by side
mask_prot = df_pp["Psite"].isna()
print(f"Protein rows:     {mask_prot.sum()}")
print(f"Phosphosite rows: {(~mask_prot).sum()}")
print()
print("--- Protein rows (first 3) ---")
print(df_pp[mask_prot].head(3).to_string())
print()
print("--- Phosphosite rows (first 3) ---")
print(df_pp[~mask_prot].head(3).to_string())


## 2 · load_site_data() return values

In [ ]:
from phoscrosstalk.data_loader import load_site_data

timepoints = list(range(1, 15))   # 14 time points

sites, proteins, site_prot_idx, positions, t, Y, A_data, A_proteins = \
    load_site_data(SAMPLE_DIR / "protephospho.csv", timepoints)

print("Return values from load_site_data():")
print(f"  sites         : {sites}  (list[str], length N={len(sites)})")
print(f"  proteins      : {proteins}  (list[str], length K={len(proteins)})")
print(f"  site_prot_idx : {site_prot_idx}  (ndarray shape ({len(site_prot_idx)},))")
print(f"  positions     : {positions}  (ndarray shape ({len(positions)},))")
print(f"  t             : {t}  (ndarray shape ({len(t)},))")
print(f"  Y             : shape {Y.shape}  → (N_sites × T) phosphosite intensities")
print(f"  A_data        : shape {A_data.shape}  → (K × T) protein abundances")
print(f"  A_proteins    : {A_proteins}")


In [ ]:
print("site_prot_idx mapping:")
for s, idx in zip(sites, site_prot_idx):
    print(f"  {s:<18} → protein index {idx}  ({proteins[idx]})")


## 3 · load_rna_data()

In [ ]:
from phoscrosstalk.data_loader import load_rna_data

gene_ids, t_rna, rna_matrix = load_rna_data(
    SAMPLE_DIR / "mrna.csv",
    timepoints=list(range(1, 15))
)

print(f"gene_ids   : {gene_ids}")
print(f"t_rna      : {t_rna}  shape {t_rna.shape}")
print(f"rna_matrix : shape {rna_matrix.shape}  → (K × T_rna) mRNA levels")


## 4 · kinase_sites.tsv and tf_mrna.csv

In [ ]:
df_kin = pd.read_csv(SAMPLE_DIR / "kinase_sites.tsv", sep="\t")
print("kinase_sites.tsv  shape:", df_kin.shape)
print("Columns:", list(df_kin.columns))
df_kin


In [ ]:
df_tf = pd.read_csv(SAMPLE_DIR / "tf_mrna.csv")
print("tf_mrna.csv  shape:", df_tf.shape)
print("Columns:", list(df_tf.columns))
df_tf


## 5 · Annotated shape summary

In [ ]:
summary = {
    "Y  (phosphosite data)":   {"shape": str(Y.shape),          "rows": "N phosphosites", "cols": "T time points"},
    "A_data (protein abund.)": {"shape": str(A_data.shape),      "rows": "K proteins",     "cols": "T time points"},
    "rna_matrix (mRNA)":       {"shape": str(rna_matrix.shape),  "rows": "K genes",        "cols": "T_rna time points"},
}
for name, info in summary.items():
    print(f"  {name}: {info['shape']}")
    print(f"    rows → {info['rows']}")
    print(f"    cols → {info['cols']}")


## 6 · Quick visualisation of loaded data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Phosphosite trajectories
ax = axes[0]
for i, s in enumerate(sites[:3]):
    ax.plot(t, Y[i], marker="o", markersize=3, label=s)
ax.set_title("Phosphosite intensities (first 3)")
ax.set_xlabel("time index")
ax.set_ylabel("intensity")
ax.legend(fontsize=7)

# Protein abundance trajectories
ax = axes[1]
for k in range(A_data.shape[0]):
    ax.plot(t, A_data[k], marker="s", markersize=3, label=proteins[k])
ax.set_title("Protein abundances")
ax.set_xlabel("time index")
ax.legend(fontsize=8)

# mRNA trajectories
ax = axes[2]
for g, gene in enumerate(gene_ids):
    ax.plot(t_rna, rna_matrix[g], marker="^", markersize=3, label=gene)
ax.set_title("mRNA levels")
ax.set_xlabel("time index")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_raw_trajectories.png", dpi=100)
plt.show()
